# Critical Habitat (English) Exploration

**Objective:** Perform initial inspection and summary of the English critical habitat dataset for Species at Risk in Canada.

**Data source(s):**
- Dataset page: https://open.canada.ca/data/en/dataset/47caa405-be2b-4e9e-8f53-c478ade2ca74
- English ZIP: https://data-donnees.az.ec.gc.ca/api/file?path=%2Fspecies%2Fprotectrestore%2Fcritical-habitat-species-at-risk-canada%2FCriticalHabitat.zip

**Date/author:** 2026-03-11, Peter


In [ ]:
from pathlib import Path
import zipfile
import pandas as pd

try:
    import geopandas as gpd
    HAS_GEOPANDAS = True
except ImportError:
    HAS_GEOPANDAS = False

print(f"GeoPandas available: {HAS_GEOPANDAS}")


In [ ]:
project_root = Path("..")
english_zip = project_root / "datasets" / "raw" / "CriticalHabitat.zip"

print(f"Expected English ZIP path: {english_zip.resolve()}")
print(f"File exists: {english_zip.exists()}")


In [ ]:
zip_listing_df = pd.DataFrame()

if english_zip.exists():
    with zipfile.ZipFile(english_zip, "r") as zf:
        names = zf.namelist()
    zip_listing_df = pd.DataFrame({"archive_path": names})
    zip_listing_df["extension"] = zip_listing_df["archive_path"].str.rsplit(".", n=1).str[-1].str.lower()
    display(zip_listing_df.head(20))
    print(f"Total files in ZIP: {len(zip_listing_df)}")
else:
    print("English ZIP not found yet. Download it to datasets/raw/CriticalHabitat.zip and rerun.")


In [ ]:
gdf = None

if english_zip.exists() and HAS_GEOPANDAS:
    with zipfile.ZipFile(english_zip, "r") as zf:
        candidates = [n for n in zf.namelist() if n.lower().endswith((".gpkg", ".shp"))]

    if candidates:
        first_layer = candidates[0]
        print(f"Attempting to read layer: {first_layer}")
        zip_uri = f"zip://{english_zip.resolve()}!{first_layer}"
        try:
            gdf = gpd.read_file(zip_uri)
            print(f"Loaded rows: {len(gdf)}")
            print(f"Columns: {len(gdf.columns)}")
            display(gdf.head())
        except Exception as exc:
            print(f"Could not read spatial layer from ZIP: {exc}")
    else:
        print("No .gpkg or .shp found in ZIP listing.")
elif not HAS_GEOPANDAS:
    print("GeoPandas is not installed. Install geopandas to run spatial analysis cells.")


In [ ]:
if gdf is not None:
    summary = pd.DataFrame({
        "column": gdf.columns,
        "dtype": [str(gdf[col].dtype) for col in gdf.columns],
        "null_count": [int(gdf[col].isna().sum()) for col in gdf.columns]
    })
    display(summary)
else:
    print("No loaded GeoDataFrame yet; summary will appear after successful spatial load.")


### Next Steps
1. Download `CriticalHabitat.zip` to `datasets/raw/`.
2. Confirm layer format(s) and projection.
3. Add topic-specific analyses (species counts, province/territory breakdown, and area summaries).


---
### Open In Colab
[Open this notebook in Google Colab](https://colab.research.google.com/github/Peter/ICD2O-OSC-2026-Project-Plan-Team-2/blob/main/notebooks/critical-habitat-english-exploration.ipynb)
